In [3]:
import pandas as pd
import torch
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.pool import NullPool

load_dotenv(override=True)

db_user = os.getenv('bbstats_db_user')
db_pass = os.getenv('bbstats_db_password')
db_host = os.getenv('bbstats_db_host')
db_name = os.getenv('bbstats_db_name')

engine = create_engine(
    f'mysql+mysqlconnector://{db_user}:{db_pass}@{db_host}/{db_name}?charset=utf8mb4',
    echo=False,
    poolclass=NullPool,
)

In [4]:
sql = f"""

    SELECT 
        pd.fullName,
        ps.era_plus,
        ps.dice_plus,
        ps.k_pct_plus,
        ps.bb_pct_plus,
        ps.hr9_plus
    from player_details as pd
    inner join `starting_pitcher_similarity_profiles` as ps
        ON
            pd.playerID = ps.playerID
    where
        pd.debutDate >= '1980-01-01'
        and pd.position = 'SP'

"""

with engine.connect() as conn:
    df = pd.read_sql(sql, conn)

df.head()


,fullName,era_plus,dice_plus,k_pct_plus,bb_pct_plus,hr9_plus
0,Andrew Abbott,126.130700,101.273561,104.555526,97.815433,97.039980
1,Jim Abbott,92.407803,94.195502,81.383737,95.483740,102.828720
2,Paul Abbott,112.523433,92.078455,95.425807,75.668721,101.958229
3,Sandy Alcantara,121.374137,112.652363,94.084093,116.334937,134.370375
4,Tyler Alexander,92.221018,86.521487,80.744929,131.548669,78.083865


In [5]:
features = torch.tensor(df[['dice_plus', 'k_pct_plus', 'bb_pct_plus', 'hr9_plus']].values, dtype=torch.float32)
target = torch.tensor(df[['era_plus']].values, dtype=torch.float32)

print(features.shape)
print(target.shape)

torch.Size([806, 4])
torch.Size([806, 1])


In [6]:


X_train, X_test = features[:643], features[643:]
Y_train, Y_test = target[:643], target[643:]


X_test.shape

torch.Size([163, 4])

In [7]:
import torch.nn as nn

class PitcherModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 16),  # 4 features in, 16 neurons
            nn.ReLU(),
            nn.Linear(16, 1)   # 16 neurons in, 1 output (ERA+)
        )
    
    def forward(self, x):
        return self.network(x)

model = PitcherModel()
print(model)

PitcherModel(
  (network): Sequential(
    (0): Linear(in_features=4, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [8]:
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [9]:
epochs = 1000

for epoch in range(epochs):
    # forward pass
    predictions = model(X_train)
    
    # calculate loss
    loss = loss_fn(predictions, Y_train)
    
    # backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # print progress every 10 epochs
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 11792.9980
Epoch 10, Loss: 3457.2610
Epoch 20, Loss: 245.2305
Epoch 30, Loss: 318.9447
Epoch 40, Loss: 197.8877
Epoch 50, Loss: 96.2891
Epoch 60, Loss: 108.7004
Epoch 70, Loss: 89.0150
Epoch 80, Loss: 87.3189
Epoch 90, Loss: 87.1421
Epoch 100, Loss: 85.9172
Epoch 110, Loss: 85.3544
Epoch 120, Loss: 85.0253
Epoch 130, Loss: 84.6956
Epoch 140, Loss: 84.3896
Epoch 150, Loss: 84.1167
Epoch 160, Loss: 83.8690
Epoch 170, Loss: 83.6435
Epoch 180, Loss: 83.4422
Epoch 190, Loss: 83.2640
Epoch 200, Loss: 83.1077
Epoch 210, Loss: 82.9692
Epoch 220, Loss: 82.8455
Epoch 230, Loss: 82.7340
Epoch 240, Loss: 82.6312
Epoch 250, Loss: 82.5371
Epoch 260, Loss: 82.4506
Epoch 270, Loss: 82.3702
Epoch 280, Loss: 82.2947
Epoch 290, Loss: 82.2242
Epoch 300, Loss: 82.1574
Epoch 310, Loss: 82.0900
Epoch 320, Loss: 82.0233
Epoch 330, Loss: 81.9582
Epoch 340, Loss: 81.8935
Epoch 350, Loss: 81.8296
Epoch 360, Loss: 81.7688
Epoch 370, Loss: 81.7131
Epoch 380, Loss: 81.6586
Epoch 390, Loss: 81.6037
Ep

In [10]:
model.eval()
with torch.no_grad():
    test_predictions = model(X_test)
    test_loss = loss_fn(test_predictions, Y_test)
    print(f"Test Loss: {test_loss.item():.4f}")

Test Loss: 107.8874


In [12]:
# get your original dataframe index for test pitchers
test_df = df.iloc[643:].copy()
test_df['predicted_era_plus'] = test_predictions.detach().numpy()
test_df['error'] = test_df['predicted_era_plus'] - test_df['era_plus']

print(test_df[['fullName', 'era_plus', 'predicted_era_plus', 'error']].to_string())

                    fullName    era_plus  predicted_era_plus      error
643           Curt Schilling  123.796585          121.649620  -2.146965
644            Jason Schmidt  106.205103          109.759003   3.553900
645            Pete Schourek   88.943554           93.895363   4.951809
646               Ken Schrom   89.728999           87.372284  -2.356715
647    Spencer Schwellenbach  131.767200          132.094910   0.327710
648            Scott Scudder   80.565737           86.684570   6.118833
649                 JP Sears   90.418073           90.331863  -0.086210
650               Aaron Sele   86.596919           86.134277  -0.462642
651              Kodai Senga  146.194135          114.313118 -31.881017
652        Antonio Senzatela   79.234797           94.927704  15.692907
653            Jae Weong Seo   97.036706           91.288452  -5.748254
654            Luis Severino  120.287307          121.939224   1.651917
655               Ben Sheets  101.187667          105.501892   4

## Notes

- Started by getting all the era adjusted data for current pitchers
- created feature and target tensors. feature are going to be used to make the inference, target is era+ which is what we're going to predict
- split into test and train groups